In [1]:
%pip install lightgbm catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # one-liner charts, high-level
import plotly.graph_objects as go # full control chart

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

import optuna

In [ ]:
TARGET = "Exited"

categorical_columns = ["Geography", "Gender"]
numerial_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
]

drop_columns  = ["id", "CustomerId"]
submission_columns = ["id", "Exited"]

In [ ]:
def load_data():
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

    train_df['source'] = 'train'
    test_df['source'] = 'test'

    df = pd.concat([train_df, test_df], ignore_index=True)

    return train_df, test_df, df

In [ ]:
def data_spliter(df):
    train = df[df['source'] == 'train'].drop(columns=drop_columns)
    submission_df = df[df['source'] == 'test'][submission_columns]

    X_train = train_df.drop(columns=[TARGET])
    y_train = train_df[TARGET]

    return X_train, y_train, submission_df

In [ ]:
train_df, test_df, df = load_data()

# EDA

In [ ]:
len(df)

-   **Customer** ID: Уникальный идентификатор каждого клиента.
-   **Surname**: Фамилия клиента.
-   **Credit Score**: Числовое значение, представляющее кредитный рейтинг клиента.
-   **Geography**: Страна проживания клиента (Франция, Испания или Германия).
-   **Gender**: Пол клиента (Мужской или Женский).
-   **Age**: Возраст клиента.
-   **Tenure**: Количество лет, которое клиент обслуживается в банке.
-   **Balance**: Баланс на счёте клиента.
-   **NumOfProducts**: Количество банковских продуктов, которыми пользуется клиент (например, сберегательный счёт, кредитная карта).
-   **HasCrCard**: Наличие кредитной карты у клиента (1 = да, 0 = нет).
-   **IsActiveMember**: Является ли клиент активным членом банка (1 = да, 0 = нет).
-   **EstimatedSalary**: Предполагаемая заработная плата клиента.
-   **Exited**: Ушёл ли клиент (1 = да, 0 = нет).


In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isna().sum()

In [ ]:
df.head()

## Pairplot

In [ ]:
fig = sns.pairplot(
    data=train_df[numerial_columns + [TARGET]],
    hue=TARGET,
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "s": 15, "edgecolor": None},
    diag_kws={"fill": True, "alpha": 0.6},
    palette={0: "#2196F3", 1: "#FF5722"},
    # corner=True,
)
fig.figure.suptitle("Pairplot of Numerical Features by Churn Status", y=1.02, fontsize=16, fontweight="bold")

## Surnames

In [ ]:
display(df['Surname'].value_counts().reset_index())

In [ ]:
df[df['Surname'].str.endswith('ov') | df['Surname'].str.endswith('ova') | df['Surname'].str.endswith('ev') | df['Surname'].str.endswith('eva')]

In [ ]:
df[df['Surname'].str.contains("?", regex=False)]

In [ ]:
display(df[df['Surname'].str.contains("'", regex=False)])

## Balance

In [ ]:
plt.figure(figsize=(16,9))
sns.histplot(
    data=train_df,
    x="Balance",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Salary

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=train_df,
    x="EstimatedSalary",
    hue='Exited',
    bins=50,
    multiple='fill',
)

## Age

In [ ]:
plt.figure(figsize=(12,5))
ax = sns.countplot(
    data=train_df,
    x=df['Age'].astype(int),
    hue='Exited',
)

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(
    data=train_df,
    x="Age",
    hue='Exited',
    bins=60,
    multiple='fill',
)
plt.xticks(np.arange(18,75))
plt.tight_layout()

In [ ]:
corr_df = pd.get_dummies(
        data=df.drop(columns=drop_columns + ["source"] + ["Surname"]),
        columns=["Gender", "Geography"],
    ).corr()

In [ ]:
np.triu(np.ones_like(corr_df, dtype=bool))

In [ ]:
corr_df = pd.get_dummies(
        data=df.drop(columns=drop_columns + ["source"] + ["Surname"]),
        columns=["Gender", "Geography"],
    ).corr()

corr_mask = np.triu(np.ones_like(corr_df, dtype=bool))

plt.figure(figsize=(16,9))
sns.heatmap(
    data=corr_df,
    mask=corr_mask,
    annot=True,
    fmt='.2f',
    cmap="coolwarm",
    square=True,
    center=0
)


In [ ]:
# Compute correlation matrix with one-hot encoded categoricals
corr_df = pd.get_dummies(
    data=df.drop(columns=drop_columns + ["source"] + ["Surname"]),
    columns=["Gender", "Geography"],
).corr()

# Sort features by absolute correlation with target for readability
target = "Exited"
target_corr = corr_df[TARGET].drop(TARGET).abs().sort_values(ascending=False)
sorted_features = [TARGET] + target_corr.index.tolist()
corr_sorted = corr_df.loc[sorted_features, sorted_features]

# --- Plot 1: Full correlogram (sorted by target correlation) ---
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

mask = np.triu(np.ones_like(corr_sorted, dtype=bool), k=1)
sns.heatmap(
    corr_sorted,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8},
    ax=axes[0],
)
axes[0].set_title("Feature Correlation Matrix\n(sorted by |corr| with Exited)", fontsize=14, fontweight="bold")

# --- Plot 2: Bar chart of correlations with target ---
target_corr_signed = corr_df[TARGET].drop(TARGET).reindex(target_corr.index)
colors = ["#FF5722" if v > 0 else "#2196F3" for v in target_corr_signed]

axes[1].barh(target_corr_signed.index[::-1], target_corr_signed.values[::-1], color=colors[::-1], edgecolor="white")
axes[1].set_xlabel("Pearson Correlation", fontsize=12)
axes[1].set_title(f"Feature Correlation with '{target}'\n(red = positive, blue = negative)", fontsize=14, fontweight="bold")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()